# Quantum Adders

Quantum adders are quantum implementations of classical addition operation - circuits that add two quantum numbers $a$ and $b$ and either store the result in a new register (out-of-place adders) or overwrite one of the addends with it (in-place adders).

In the Arithmetic Data Types kata we learned to implement the so-called "naive" adder based on incrementing the first addend $a$ in-place controlled on each digit of the second addend $b$.
In this kata, we will explore more efficient adders.

> Here, we will focus on implementing modular adders for unsigned integers only. The implementations of adders provided by Workbench are general and work for any arithmetic data types, but using unsigned integers will allow us to focus on the main ideas behind each adder without going through the minutiae of dealing with sign extension and fixed-point representation. The implementation of a modular adder can also be modified to handle non-modular addition if you choose to keep the carry from adding the most significant qubits as the most significant bit of the result.

We will consider only in-place adders which will modify the first addend $a$ to store the sum. An in-place adder can be easily turned into an out-of-place adder by copying the addend $a$ into a new register first and then using it as the target for in-place addition.

**This kata covers the following topics:**

- Ripple-carry adder - an adder that implements textbook addition using a sequence of steps to compute sum and carry.
- Cuccaro adder - a qubit-optimal adder that uses in-place majority calculation, described in "A new quantum ripple-carry addition circuit" ([arXiv:quant-ph/0410184](https://arxiv.org/abs/quant-ph/0410184)).

**What you should know to start working on this kata:**

- Fundamental quantum concepts
- Controlled gates
- Arithmetic data types, in particular unsigned integer representation using little-endian notation

## Part 1. Ripple-carry adder

Ripple-carry adder implements the "textbook" algorithm for classical addition. It performs addition digit-by-digit, starting with the least significant digits, and on each step it computes sum and carry bits of pairs of digits and carry bits from previous steps. The algorithm is very straightforward, if inefficient. Let's see how to implement it!

### Problem 1.1. In-place sum of two bits

**Inputs:** 

1. A QUInt register $a$ of length $1$ in an arbitrary superposition state $\ket{a}$.
2. A QUInt register $b$ of length $1$ in an arbitrary superposition state $\ket{b}$.

**Goals:**

* Transform the state of $a$ into the lowest bit of the sum of $a$ and $b$.
* Leave the state of $b$ unchanged.

$$\ket{a}\ket{b} \rightarrow \ket{(a + b) \textrm{ mod } 2} \ket{b}$$

> This function is a building block for larger adders, but you can also think of it as an adder of single-bit numbers modulo $2$.

In [ ]:
from psiqdk.workbench import QUInt
from test_Adders import problem

@problem
def sum_two_bits(a: QUInt, b: QUInt) -> None:
    # Write your code here
    ...

### Problem 1.2. Carry of two bits

**Inputs:** 

1. A QUInt register $a$ of length $1$ in an arbitrary superposition state $\ket{a}$.
2. A QUInt register $b$ of length $1$ in an arbitrary superposition state $\ket{b}$.
3. A QUInt register $c$ of length $1$ in the state $\ket{0}$.

**Goals:**

* Transform the state of $c$ into the carry bit of the sum of bits $a$ and $b$.
* Leave the states of $a$ and $b$ unchanged.

$$\ket{a}\ket{b}\ket{0} \rightarrow \ket{a} \ket{b} \ket{\textrm{carry}(a, b)}$$

In [ ]:
from psiqdk.workbench import QUInt
from test_Adders import problem

@problem
def carry_two_bits(a: QUInt, b: QUInt, c: QUInt) -> None:
    # Write your code here
    ...

### Problem 1.3. In-place sum of three bits

**Inputs:** 

1. A QUInt register $a$ of length $1$ in an arbitrary superposition state $\ket{a}$.
2. A QUInt register $b$ of length $1$ in an arbitrary superposition state $\ket{b}$.
3. A QUInt register $c$ of length $1$ in an arbitrary superposition state $\ket{c}$.

**Goals:**

* Transform the state of $a$ into the lowest bit of the sum of $a$, $b$, and $c$.
* Leave the states of $b$ and $c$ unchanged.

$$\ket{a}\ket{b}\ket{c} \rightarrow \ket{(a + b + c) \textrm{ mod } 2} \ket{b}\ket{c}$$

In [ ]:
from psiqdk.workbench import QUInt
from test_Adders import problem

@problem
def sum_three_bits(a: QUInt, b: QUInt, c: QUInt) -> None:
    # Write your code here
    ...

### Problem 1.4. Carry of three bits

**Inputs:** 

1. A QUInt register $a$ of length $1$ in an arbitrary superposition state $\ket{a}$.
2. A QUInt register $b$ of length $1$ in an arbitrary superposition state $\ket{b}$.
3. A QUInt register $c$ of length $1$ in an arbitrary superposition state $\ket{c}$.
4. A QUInt register $d$ of length $1$ in the state $\ket{0}$.

**Goals:**

* Transform the state of $d$ into the carry bit of the sum of bits $a$, $b$, and $c$.
* Leave the states of $a$, $b$, and $c$ unchanged.

$$\ket{a}\ket{b}\ket{c}\ket{0} \rightarrow \ket{a} \ket{b} \ket{c} \ket{\textrm{carry}(a, b, c)}$$

In [ ]:
from psiqdk.workbench import QUInt
from test_Adders import problem

@problem
def carry_three_bits(a: QUInt, b: QUInt, c: QUInt, d: QUInt) -> None:
    # Write your code here
    ...

### Problem 1.5. Two-bit ripple-carry adder

**Inputs:** 

1. A QUInt register $a$ of length $2$ in an arbitrary superposition state $\ket{a}$.
2. A QUInt register $b$ of length $2$ in an arbitrary superposition state $\ket{b}$.

**Goal:** 

* Transform the state of $a$ into the sum of $a$ and $b$ modulo $2^2$.
* Leave the state of $b$ unchanged.

$$\ket{a}\ket{b} \rightarrow \ket{(a + b) \textrm{ mod } 2^2} \ket{b}$$

As usual in Workbench, all numbers in this and subsequent tasks are stored in little-endian notation: the least significant bit is stored first.

You can allocate $1$ auxiliary qubit for your adder. Make sure to return it to the $\ket{0}$ state and release it as part of the computation.

> Note that for this task we'll switch to implementing the solutions as Qubricks instead of functions. You'll need to allocate auxiliary qubits to store the intermediary results of your computation, and the Qubricks framework offers convenient methods for managing auxiliary qubots.
>
> Implementing a Qubrick solution is very similar to implementing a simple function: you're given a skeleton class with one or several methods to implement. The main method of a Qubrick class is `_compute`: this is the method you'll call to run the computation performed by this Qubrick in place of a function call.
>
> You can read more about implementing Qubricks in the [Qubricks tutorial](https://construct.psiquantum.com/docs/psiqdk-workbench/new-tutorials/Qubricks.html) and the [Auxiliary Qubit Management in Qubricks tutorial](https://construct.psiquantum.com/docs/psiqdk-workbench/new-tutorials/Qubricks-Qubit-Management.html).

In [ ]:
from psiqdk.workbench import QUInt, Qubrick
from test_Adders import problem

@problem
class RippleCarryAdderTwoBit(Qubrick):
    def _compute(self, a: QUInt, b: QUInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        # Write your code here
        ...

### Problem 1.6. Ripple-carry adder

**Inputs:** 

1. A QUInt register $a$ of length $N \ge 2$ in an arbitrary superposition state $\ket{a}$.
2. A QUInt register $b$ of length $N$ in an arbitrary superposition state $\ket{b}$.

**Goal:** 

* Transform the state of $a$ into the sum of $a$ and $b$ modulo $2^N$.
* Leave the state of $b$ unchanged.

$$\ket{a}\ket{b} \rightarrow \ket{(a + b) \textrm{ mod } 2^N} \ket{b}$$

You can allocate $N - 1$ auxiliary qubits for your adder. Make sure to return them to the $\ket{0}$ state and release them as part of the computation. 

In [ ]:
from psiqdk.workbench import QUInt, Qubrick
from test_Adders import problem

@problem
class RippleCarryAdder(Qubrick):
    def _compute(self, a: QUInt, b: QUInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        # Write your code here
        ...

## Part 2. Cuccaro adder

Cuccaro adder implements an improved algorithm which allows you to do addition using just one auxiliary qubit.

### Problem 2.1. Majority gate

**Inputs:** 

1. A QUInt register $a$ of length $1$ in an arbitrary superposition state $\ket{a}$.
2. A QUInt register $b$ of length $1$ in an arbitrary superposition state $\ket{b}$.
3. A QUInt register $c$ of length $1$ in an arbitrary superposition state $\ket{c}$.

**Goal:**

Implement in-place majority gate that has the following effect:

$$\ket{a}\ket{b}\ket{c} \rightarrow \ket{a \oplus b} \ket{\textrm{carry}(a, b, c)} \ket{b \oplus c}$$

In [ ]:
from psiqdk.workbench import QUInt
from test_Adders import problem

@problem
def maj(a: QUInt, b: QUInt, c: QUInt) -> None:
    # Write your code here
    ...

### Problem 2.2. UnMajority and Add gate

**Inputs:** 

1. A QUInt register $a$ of length $1$ in an arbitrary superposition state $\ket{a \oplus b}$.
2. A QUInt register $b$ of length $1$ in an arbitrary superposition state $\ket{\textrm{carry}(a, b, c)}$.
3. A QUInt register $c$ of length $1$ in an arbitrary superposition state $\ket{b \oplus c}$.

> The input states for this problem are just the states of the registers $\ket{a} \ket{b} \ket{c}$ after applying to them the majority gate from the previous problem.

**Goal:**

Implement in-place "unmajority and add" gate that has the following effect:

$$\ket{a \oplus b} \ket{\textrm{carry}(a, b, c)} \ket{b \oplus c} \rightarrow \ket{a \oplus b \oplus c} \ket{b} \ket{c}$$

In [ ]:
from psiqdk.workbench import QUInt
from test_Adders import problem

@problem
def uma(a: QUInt, b: QUInt, c: QUInt) -> None:
    # Write your code here
    ...

### Problem 2.3. One-bit Cuccaro adder

**Inputs:** 

1. A QUInt register $a$ of length $1$ in an arbitrary superposition state $\ket{a}$.
2. A QUInt register $b$ of length $1$ in an arbitrary superposition state $\ket{b}$.

**Goals:**

* Transform the state of $a$ into the lowest bit of the sum of $a$ and $b$.
* Leave the state of $b$ unchanged.

$$\ket{a}\ket{b} \rightarrow \ket{(a + b) \textrm{ mod } 2} \ket{b}$$

You can allocate $1$ auxiliary qubit for your adder. Make sure to return it to the $\ket{0}$ state and release it as part of the computation.

> This is the same task as in problem 1.1; this time, implement it using only the primitives MAJ and UMA without any other gates.

In [ ]:
from psiqdk.workbench import QUInt, Qubrick
from test_Adders import problem

@problem
class CuccaroAdderOneBit(Qubrick):
    def _compute(self, a: QUInt, b: QUInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        # Write your code here
        ...

### Problem 2.4. Two-bit Cuccaro adder

**Inputs:** 

1. A QUInt register $a$ of length $2$ in an arbitrary superposition state $\ket{a}$.
2. A QUInt register $b$ of length $2$ in an arbitrary superposition state $\ket{b}$.

**Goals:**

* Transform the state of $a$ into the sum of $a$ and $b$ modulo $2^2$.
* Leave the state of $b$ unchanged.

$$\ket{a}\ket{b} \rightarrow \ket{(a + b) \textrm{ mod } 2^2} \ket{b}$$

You can allocate $1$ auxiliary qubit for your adder. Make sure to return it to the $\ket{0}$ state and release it as part of the computation.

> This is the same task as in problem 1.5; again, implement it using only the primitives MAJ and UMA without any other gates.

In [ ]:
from psiqdk.workbench import QUInt, Qubrick
from test_Adders import problem

@problem
class CuccaroAdderTwoBit(Qubrick):
    def _compute(self, a: QUInt, b: QUInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        # Write your code here
        ...

### Problem 2.5. Cuccaro adder

**Inputs:** 

1. A QUInt register $a$ of length $N \ge 2$ in an arbitrary superposition state $\ket{a}$.
2. A QUInt register $b$ of length $N$ in an arbitrary superposition state $\ket{b}$.

**Goals:**

* Transform the state of $a$ into the sum of $a$ and $b$ modulo $2^N$.
* Leave the state of $b$ unchanged.

$$\ket{a}\ket{b} \rightarrow \ket{(a + b) \textrm{ mod } 2^N} \ket{b}$$

You can allocate $1$ auxiliary qubit for your adder. Make sure to return it to the $\ket{0}$ state and release it as part of the computation.

> This is the same task as in problem 1.6; again, implement it using only the primitives MAJ and UMA without any other gates.

In [ ]:
from psiqdk.workbench import QUInt, Qubrick
from test_Adders import problem

@problem
class CuccaroAdder(Qubrick):
    def _compute(self, a: QUInt, b: QUInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        # Write your code here
        ...

## Conclusion

Congratulations! In this kata you've learned to implement several quantum adders with different qubits and gates resource requirements.

> Copyright (c) 2026 PsiQuantum